# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Classification and Ranking (Scoring)**

We frame this as a **binary classification task** combined with **scoring/ranking**. The model classifies whether a content item is entering a traffic decay trajectory ($1$) or remaining stable/growing ($0$). Furthermore, it outputs a continuous probability score, allowing us to generate a prioritized, ranked queue of pages that require an urgent editorial refresh.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Total dataset shape for framing: {df.shape}")

Total dataset shape for framing: (30000, 44)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target Label:** A binary indicator `is_declining_label` ($1$ if `trend_direction == "down"`, $0$ otherwise).

**Origin:** It is an **observed outcome** computed from historical search performance trajectories over a defined evaluation window, comparing recent traffic periods against previous periods.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
print("Class distribution of target label:")
print(df['is_declining_label'].value_counts(normalize=True))

Class distribution of target label:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K (specifically Precision@50)**

In content optimization, stakeholders only have the editorial bandwidth to inspect a limited number of pages (e.g., the top 50 flagged items). **Precision@50** measures what proportion of the model's top 50 recommendations are actually declining. A 'good' model significantly outperforms the random base rate and our hand-written rule baseline.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
base_rate = df['is_declining_label'].mean()
print(f"Dataset base rate for declining pages: {base_rate:.3f}")

Dataset base rate for declining pages: 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = One unique content page (`content_id`) belonging to a specific client (`client_id`).**

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols_to_show = ['content_id', 'client_id', 'content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'trend_direction']
display(df[cols_to_show].head(3))

,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,187,20,3803,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,36.5,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement relies on rigid thresholds (e.g., *update if days since last update > 180 AND impressions > 500*). However, traffic decay is governed by complex, non-linear interactions across multiple dimensions—such as how position volatility offsets content staleness differently for high-volume versus low-volume queries. Machine learning algorithms can automatically capture these multi-variable interactions and weigh continuous risk signals without arbitrary hardcoded thresholds.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature framing verification complete. Ready for modeling pipelines.")

Feature framing verification complete. Ready for modeling pipelines.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.